# Rates environment

A `RatesEnvironment` is one anchor date and five slots: forward, domestic discount, foreign
discount, volatility, and fixings. An empty slot is unset. This notebook fills all five from
the sample curve, compares a deterministic forward with one that can roll, and builds the same
outright from two discount curves.


In [ ]:
import logging
import math
from datetime import date

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

import sjd
from sjd import examples

sjd.set_log_level("INFO")

anchor = examples.ANCHOR
pillars = [anchor, date(2026, 12, 16), date(2027, 6, 16), date(2027, 9, 16)]
forwards, discount_factors, volatilities = examples.fx_curve_values(pillars)
foreign_factors = [
    math.exp(-examples.FOREIGN_RATE * examples.year_fraction(day)) for day in pillars
]
discount = sjd.DiscountCurve(anchor, pillars, discount_factors)
foreign = sjd.DiscountCurve(anchor, pillars, foreign_factors)
volatility = sjd.VolatilityCurve(anchor, pillars, volatilities)
fixings = sjd.DailyFixingCurve([anchor], [examples.SPOT])
forward = sjd.ForwardCurve(
    anchor,
    pillars,
    forwards,
    roll=sjd.ForwardRoll.FORWARD_VOLATILITY,
    volatility=volatility,
)

environment = sjd.RatesEnvironment(anchor)
environment.set_frs(forward)
environment.set_drs(discount)
environment.set_foreign_drs(foreign)
environment.set_vrs(volatility)
environment.set_fixing_rs(fixings)
sjd.require_valid(environment)
assert math.isclose(
    environment.foreign_drs().get_discount_factor(pillars[-1]),
    foreign.get_discount_factor(pillars[-1]),
)
print(environment)


## Deterministic and stochastic forwards

The same pillars with `ForwardRoll.DETERMINISTIC` and `ForwardRoll.FORWARD_VOLATILITY` have
the same forward, and a forward contract prices to the same value on both. A path generator
can roll only the stochastic one. The deterministic evolver reports
`PATH_GENERATOR_FORWARD_NOT_STOCHASTIC` and stops before any path is drawn. Drawing the paths
is notebook 10.


In [ ]:
deterministic = sjd.ForwardCurve(
    anchor, pillars, forwards, roll=sjd.ForwardRoll.DETERMINISTIC, volatility=volatility
)
horizon = pillars[-1]
settlement = date(2027, 3, 16)
contract = sjd.EuropeanForward(settlement, 17.0, 1_000_000.0, sjd.BuySell.BUY)

def priced(curve):
    market = sjd.RatesEnvironment(anchor)
    market.set_frs(curve)
    market.set_drs(discount)
    market.set_vrs(volatility)
    market.set_fixing_rs(fixings)
    sjd.require_valid(contract, market)
    return contract.get_value(market)

def evolver_report(curve):
    market = sjd.RatesEnvironment(anchor)
    market.set_frs(curve)
    market.set_drs(discount)
    market.set_vrs(volatility)
    market.set_fixing_rs(fixings)
    generator = sjd.SingleFactorEconomicScenarioGenerator(market)
    evolver = sjd.PortfolioEvolver(
        sjd.Portfolio([contract]), generator, [anchor, settlement]
    )
    return evolver.validate()

assert math.isclose(deterministic.get_forward(horizon), forward.get_forward(horizon))
assert math.isclose(priced(deterministic), priced(forward))
blocked = evolver_report(deterministic)
ready = evolver_report(forward)
assert not blocked.ok
assert blocked.issues[0].code == "PATH_GENERATOR_FORWARD_NOT_STOCHASTIC"
assert ready.ok
print(f"value {priced(forward):,.2f}")
blocked


## FX from two discount curves

`ForwardCurveFX` is a spot and two discount curves. Passing a spot date moves the spot to the
anchor by the two discount factors. Passing the spot already on the anchor leaves it there, and
the forward is `spot * foreign discount / domestic discount`.

`ForwardCurveDeterministicFX` is that curve with a deterministic roll.
`ForwardCurveStochasticFX` adds a volatility so an evolver can drive it. The forward level is
the same on all three. `foreign_drs()` reads the foreign curve back.


In [ ]:
spot_date = date(2026, 9, 18)
moved = sjd.ForwardCurveFX(anchor, examples.SPOT, spot_date, discount, foreign)
spot_at_anchor = (
    examples.SPOT
    * discount.get_discount_factor(spot_date)
    / foreign.get_discount_factor(spot_date)
)
expected_moved = (
    spot_at_anchor
    * foreign.get_discount_factor(horizon)
    / discount.get_discount_factor(horizon)
)
expected_anchor = examples.SPOT * math.exp(
    (examples.DOMESTIC_RATE - examples.FOREIGN_RATE) * examples.year_fraction(horizon)
)
on_anchor = sjd.ForwardCurveFX(anchor, examples.SPOT, discount, foreign)
deterministic_fx = sjd.ForwardCurveDeterministicFX(
    anchor, examples.SPOT, discount, foreign
)
stochastic_fx = sjd.ForwardCurveStochasticFX(
    anchor, examples.SPOT, discount, foreign, volatility
)

assert moved.validate().ok
assert math.isclose(moved.get_forward(horizon), expected_moved)
assert math.isclose(on_anchor.get_forward(horizon), expected_anchor)
assert math.isclose(deterministic_fx.get_forward(horizon), expected_anchor)
assert math.isclose(stochastic_fx.get_forward(horizon), expected_anchor)

fx_market = sjd.RatesEnvironment(anchor)
fx_market.set_frs(stochastic_fx)
fx_market.set_drs(discount)
fx_market.set_foreign_drs(foreign)
fx_market.set_vrs(volatility)
sjd.require_valid(fx_market)
assert math.isclose(
    fx_market.foreign_drs().get_discount_factor(horizon),
    foreign.get_discount_factor(horizon),
)
print(f"spot moved to the anchor {moved.get_forward(anchor):.6f}")
print(f"forward on {horizon} {on_anchor.get_forward(horizon):.6f}")


## The sample market

`examples.sample_market` builds this curve, the collar and Asian dates, and an environment.
`stochastic=True` gives the forward a forward-volatility roll. Later notebooks start from this
call. The keys below are the dates and the curves.


In [ ]:
plain = examples.sample_market()
stochastic = examples.sample_market(stochastic=True)
assert plain["anchor"] == anchor
assert set(plain) == set(stochastic)
for key in ("forward", "discount", "volatility", "fixings", "environment"):
    assert key in plain

outright = sjd.EuropeanForward(
    plain["forward_settlement"], 17.0, 1.0, sjd.BuySell.BUY
)
sjd.require_valid(outright, plain["environment"])
sjd.require_valid(outright, stochastic["environment"])
assert math.isclose(
    outright.get_value(plain["environment"]),
    outright.get_value(stochastic["environment"]),
)

def sample_evolver(market):
    generator = sjd.SingleFactorEconomicScenarioGenerator(market["environment"])
    return sjd.PortfolioEvolver(
        sjd.Portfolio([outright]), generator, market["simulation_dates"]
    ).validate()

assert sample_evolver(stochastic).ok
assert not sample_evolver(plain).ok
print("anchor", plain["anchor"])
print("simulation", ", ".join(f"{day:%Y-%m-%d}" for day in plain["simulation_dates"]))
print("curves", ", ".join(key for key in plain if key.endswith(("forward", "discount", "volatility", "fixings")) or key == "environment"))


## What can go wrong

An empty environment reports `RATES_ENVIRONMENT_NO_RATES`. A discount curve built on another
anchor reports `DISCOUNT_RATE_SOURCE_ANCHOR_MISMATCH`. A discount curve with unequal dates and
factors is reported under the environment's path: the environment absorbs the curve's report,
and the curve has absorbed the interpolator's.

`sjd.require_valid` runs `validate` and raises `ValidationError`. The exception carries the
report.


In [ ]:
empty = sjd.RatesEnvironment(anchor)
empty_report = empty.validate()
assert not empty_report.ok
assert empty_report.issues[0].code == "RATES_ENVIRONMENT_NO_RATES"
empty_report


In [ ]:
other_anchor = date(2026, 1, 2)
wrong_anchor = sjd.RatesEnvironment(anchor)
wrong_anchor.set_drs(
    sjd.DiscountCurve(other_anchor, [other_anchor, date(2027, 1, 4)], [1.0, 0.95])
)
anchor_report = wrong_anchor.validate()
assert not anchor_report.ok
assert anchor_report.issues[0].code == "DISCOUNT_RATE_SOURCE_ANCHOR_MISMATCH"
assert anchor_report.issues[0].path[0] == "rates_environment"
anchor_report


In [ ]:
nested = sjd.RatesEnvironment(anchor)
nested.set_drs(sjd.DiscountCurve(anchor, [anchor, pillars[-1]], [1.0]))
nested_report = nested.validate()
assert not nested_report.ok
assert nested_report.issues[0].code == "INTERPOLATOR_DIMENSION_MISMATCH"
assert nested_report.issues[0].path[:2] == ("rates_environment", "discount_rate_source")
assert "interpolator" in nested_report.issues[0].path

try:
    sjd.require_valid(nested)
except sjd.ValidationError as error:
    assert error.code == "VALIDATION"
    assert error.report.issues[0].code == "INTERPOLATOR_DIMENSION_MISMATCH"
    print(error.code)
else:
    raise AssertionError("require_valid should refuse the broken discount curve")
nested_report


## Save and reload

The environment pickles from its anchor and whichever of the five sources is set. A forward
priced on the restored environment matches the original. The file is temporary.


In [ ]:
import pickle
import tempfile
from pathlib import Path

scratch = tempfile.TemporaryDirectory()
path = Path(scratch.name) / "environment.pkl"
path.write_bytes(pickle.dumps(environment))
restored = pickle.loads(path.read_bytes())
held = sjd.EuropeanForward(settlement, 17.0, 1_000_000.0, sjd.BuySell.BUY)
sjd.require_valid(held, environment)
sjd.require_valid(held, restored)
assert math.isclose(held.get_value(restored), held.get_value(environment))
print("reloaded", restored)
print(f"value {held.get_value(restored):,.2f}")
